# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Survivors: Exploration with `mlcroissant`
This notebook guides you through the process of loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their unique `@id` as required by the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. All metadata properties can be accessed as attributes on the `metadata` object (not via subscripting).

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs in the dataset. We will use the `@id` of record sets and fields for all operations.

Let's enumerate all available record sets in the dataset (referenced by their `@id`), and for the primary record set, we will print its available fields and columns with their `@id` as well.

In [ ]:
# List all available record sets
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print("Available Record Sets (@id):")
for rsid in record_sets:
    print(f"  - {rsid}")

if not record_sets:
    print("No record sets listed in the top-level 'recordSet' property in metadata. Attempting to find record sets via dataset API.")
    # Try a fallback: Listing all record sets defined in this package
    all_record_sets = dataset.list_record_sets()
    record_sets = [recset['@id'] for recset in all_record_sets]
    for rsid in record_sets:
        print(f"  - {rsid}")

# For demonstration, show fields for the first record set, if any
if record_sets:
    recset_id = record_sets[0]
    print(f"\nFields in record set {recset_id}:")
    recset_schema = dataset.get_record_set_schema(recset_id)
    for f in recset_schema['field']:
        # Field is a dict with @id and optionally other components
        f_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
        print(f"  - {f_id}")
    # Optionally, print columns for the first field
    print("\nSample of columns for the first field (where present):")
    first_field = recset_schema['field'][0]
    if isinstance(first_field, dict) and 'column' in first_field:
        cols = first_field['column']
        if isinstance(cols, list):
            for c in cols:
                print(f"    - {c['@id'] if isinstance(c, dict) else c}")
        else:
            print(f"    - {cols['@id'] if isinstance(cols, dict) else cols}")
else:
    print('No record sets available in the metadata.')

## 3. Data Extraction
Load records from each record set into a pandas DataFrame for further analysis. We use the `@id` (string) of each record set as keys in our DataFrame dictionary. Please ensure your field and column references always use their exact `@id`.


In [ ]:
# List all record set @id's for extraction
if not record_sets:
    all_record_sets = dataset.list_record_sets()
    record_sets = [recset['@id'] for recset in all_record_sets]

dataframes = dict()

for recset_id in record_sets:
    print(f"Loading records from record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"  Fields (@id): {list(df.columns)} (first 3 rows below)")
    display(df.head(3))

# For subsequent analysis, select the main record set
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Selected main record set (@id): {main_record_set_id}")
    print(f"Fields (@id): {list(dataframes[main_record_set_id].columns)}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using the DataFrame. We'll select a numeric field (`@id`) from the main record set, demonstrate filtering and normalization, and perform groupby operations if a categorical field is available.
All fields referenced below use their `@id` from the schema/overview section above.

In [ ]:
# EDA: select a likely numeric field and group field based on columns
from pandas.api.types import is_numeric_dtype

if main_record_set_id:
    df = dataframes[main_record_set_id]
    numeric_field_id = None
    possible_group_field = None
    # Attempt to select numeric and categorical fields by simple heuristics
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == 'object' or df[col].dtype.name == 'category'):
            possible_group_field = col
            break
    if numeric_field_id is None:
        print("Could not auto-identify a numeric field. Please review field `@id`s and select manually.")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        # Filter records by greater than mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (first 5 rows):")
        display(filtered_df.head())
        # Normalizing the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records (first 5 values):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Groupby, if a possible group field exists
        if possible_group_field:
            print(f"\nGrouping by categorical field '@id': {possible_group_field}")
            grouped_df = filtered_df.groupby(possible_group_field)[numeric_field_id].mean()
            print(grouped_df.head())
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relation to the group field (if present) using matplotlib/seaborn.

We use the exact `@id` for axis labels for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    fig, ax = plt.subplots(1,2, figsize=(12,4))
    sns.histplot(df[numeric_field_id], kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    # Conditional boxplot by group, if group field available
    if possible_group_field:
        sns.boxplot(x=df[possible_group_field], y=df[numeric_field_id], ax=ax[1])
        ax[1].set_title(f"{numeric_field_id} by {possible_group_field}")
        ax[1].set_xlabel(possible_group_field)
        ax[1].set_ylabel(numeric_field_id)
    else:
        ax[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric field identified for visualization.')

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR^2 dataset's metadata and records using `mlcroissant`, referencing all entities by their `@id` for reproducibility and correctness.
- Explored available record sets and fields using programmatic methods.
- Loaded records into DataFrames and performed basic exploratory data analysis, including filtering and normalization.
- Visualized distributions and relationships of selected fields using their `@id`s for clarity.

**Key findings and next steps:**
- This dataset supports in-depth clinical and molecular analysis of second primary colorectal cancers. It offers columns with demographic, pathological, and biomarker variables (all discoverable by `@id`).
- For advanced analysis, researchers should refer to the Croissant schema for exact meaning and provenance of each field or column by its `@id`.

_For more information, visit the dataset's [source](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)._